# Exercises - Lesson 1.1: Math Foundations 1 - The Basics

Mirror of the code in the published lesson, runnable. Each section corresponds to one of the 4 skills (vectors / cosine / softmax / temperature) plus a final production-pipeline cell.

In [ ]:
!pip install -q numpy

In [ ]:
import numpy as np

## Skill 1 - Vectors

In [ ]:
words = {
    'king':   np.array([0.9, 0.1, 0.8, 0.0]),
    'queen':  np.array([0.9, 0.9, 0.8, 0.0]),
    'man':    np.array([0.1, 0.1, 0.5, 0.0]),
    'woman':  np.array([0.1, 0.9, 0.5, 0.0]),
    'pizza':  np.array([0.0, 0.0, 0.0, 0.9]),
}

result = words['king'] - words['man'] + words['woman']
print(f'king - man + woman = {result}')
print(f"queen              = {words['queen']}")

## Skill 2 - Cosine similarity

In [ ]:
def cosine_similarity(a, b, eps=1e-6):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + eps)

documents = {
    'Python tutorial':  np.array([0.9, 0.8, 0.1, 0.0]),
    'JavaScript guide': np.array([0.8, 0.7, 0.2, 0.0]),
    'Pasta recipe':     np.array([0.0, 0.1, 0.9, 0.8]),
    'Pizza recipe':     np.array([0.0, 0.1, 0.8, 0.9]),
    'Machine learning': np.array([0.7, 0.9, 0.3, 0.0]),
}
query = np.array([0.85, 0.75, 0.15, 0.0])

ranked = sorted(documents.items(), key=lambda kv: cosine_similarity(query, kv[1]), reverse=True)
for doc, vec in ranked:
    sim = cosine_similarity(query, vec)
    print(f'  {doc:18s} cos={sim:.3f}')

## Skill 3 - Softmax (numerically stable)

In [ ]:
def softmax(logits):
    shifted = logits - np.max(logits)
    e = np.exp(shifted)
    return e / e.sum()

tokens = ['cat', 'dog', 'fish']
logits = np.array([2.0, 1.0, 0.1])
probs = softmax(logits)
for tok, p in zip(tokens, probs):
    print(f'  {tok:5s} {p:.3f}')
print(f'  sum = {probs.sum():.6f}')

## Skill 4 - Temperature

In [ ]:
def softmax_with_temp(logits, T=1.0):
    scaled = logits / T
    shifted = scaled - np.max(scaled)
    e = np.exp(shifted)
    return e / e.sum()

tokens = ['cat', 'dog', 'fish']
logits = np.array([2.0, 1.0, 0.1])
for T in [0.1, 0.5, 1.0, 2.0]:
    p = softmax_with_temp(logits, T=T)
    bar_cat = '#' * int(p[0] * 40)
    print(f'  T={T:3.1f}  cat={p[0]:.3f}  dog={p[1]:.3f}  fish={p[2]:.3f}  {bar_cat}')

## Production pipeline - all 4 primitives in 30 lines

In [ ]:
query_vec = np.array([0.85, 0.75, 0.15, 0.0])

docs = {'py_tut': np.array([0.9,0.8,0.1,0.0]),
        'pasta':  np.array([0.0,0.1,0.9,0.8])}
top = max(docs, key=lambda k: cosine_similarity(query_vec, docs[k]))

logits = np.array([2.0, 1.0, 0.1])

def sample(logits, T=0.7):
    p = np.exp((logits - logits.max()) / T)
    p = p / p.sum()
    return np.random.choice(len(p), p=p)

np.random.seed(42)
token_idx = sample(logits, T=0.7)
print(f'Retrieved: {top}, generated token idx: {token_idx}')